In [7]:
import requests
import praw
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import yfinance as yf
import seaborn as sns
sns.set()

In [ ]:
import requests

API_KEY = "YOUR_API_KEY"

CHANNEL_NAMES = ['cnbc']
CHANNEL_IDs = {}

# Function to get channel ID by channel name
def get_channel_id_by_name(channel_name, api_key):
    url = f"https://www.googleapis.com/youtube/v3/search?part=snippet&type=channel&q={channel_name}&key={api_key}"
    
    response = requests.get(url)
    data = response.json()

    return data['items'][0]['id']['channelId']

# Get channel IDs
for channel_name in CHANNEL_NAMES:
    channel_id = get_channel_id_by_name(channel_name, API_KEY)
    CHANNEL_IDs[channel_name] = channel_id

# Print results
for channel_name, channel_id in CHANNEL_IDs.items():
    print(f"{channel_name}: {channel_id}")

cnbc: UCvJJ_dzjViJCoLf5uKUTwoA


In [2]:
# Define parameters
KEYWORDS = ['recession', 'unemployment', 'crash', 'stimulus', 'crisis', 'inflation'] # List of keywords to search for. You can include your selection of relevant keywords
START_DATE = '2020-01-01T00:00:00Z'
END_DATE = '2020-05-30T23:59:59Z'

In [3]:
# Initialize YouTube API
def initialize_youtube(api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    return youtube

# Get videos from a channel
def get_videos(youtube, channel_id, start_date, end_date):
    request = youtube.search().list(
        part='snippet',
        channelId=channel_id,
        publishedAfter=start_date,
        publishedBefore=end_date,
        maxResults=50,
        type='video'
    )
    videos = []
    while request:
        response = request.execute()
        videos.extend(response['items'])
        request = youtube.search().list_next(request, response)
    return videos

# Get comments from a video
def get_comments(youtube, video_id):
    comments = []
    request = youtube.commentThreads().list(
        part='snippet',
        videoId=video_id,
        maxResults=100
    )
    while request:
        try:
            response = request.execute()
            comments.extend(response['items'])
            request = youtube.commentThreads().list_next(request, response)
        except HttpError as e:
            if e.resp.status == 403 and 'commentsDisabled' in str(e):
                print(f"Comments are disabled for video ID: {video_id}")
                break
            else:
                raise e
    return comments

# Filter data and count keyword mentions
def filter_and_count_comments(comments, keywords, start_date, end_date):
    keyword_counts = defaultdict(int)
    start = datetime.strptime(start_date, '%Y-%m-%dT%H:%M:%SZ')
    end = datetime.strptime(end_date, '%Y-%m-%dT%H:%M:%SZ')
    current_date = start

    while current_date <= end:
        keyword_counts[current_date.strftime('%Y-%m-%d')] = 0
        current_date += timedelta(days=1)

    for comment_thread in comments:
        comment = comment_thread['snippet']['topLevelComment']['snippet']
        comment_date = datetime.strptime(comment['publishedAt'], '%Y-%m-%dT%H:%M:%SZ').strftime('%Y-%m-%d')
        comment_text = comment['textDisplay'].lower()
        if any(keyword in comment_text for keyword in keywords):
            keyword_counts[comment_date] += 1

    return keyword_counts

def plot_keyword_counts(keyword_counts, channel_name):
    dates = list(keyword_counts.keys())
    counts = list(keyword_counts.values())
    df = pd.DataFrame({'Date': dates, 'Count': counts})
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date')
    df.sort_index(inplace=True)

    plt.figure(figsize=(14, 7))  # Adjust the figure size for better readability
    ax = df.plot(kind='bar', legend=False, width = 0.8)
    ax.set_title(f'Keyword Mentions per Day in {channel_name}')
    ax.set_xlabel('Date')
    ax.set_ylabel('Count of Keywords')

    # Set x-ticks at regular intervals and rotate labels for better readability
    ax.set_xticks(range(0, len(df.index), 14))
    ax.set_xticklabels(df.index.strftime('%Y-%m-%d')[::14], rotation=45, ha='right')

    #plt.tight_layout()
    plt.show()



In [ ]:
# Main script
# THIS CELL CAN TAKE SIGNIFICANT TIME TO COMPLETE EVEN CLOSE TO 20 MINUTES.
# PLEASE BE PATIENT.

# Initialize the YouTube API client
youtube = initialize_youtube(API_KEY)

# Fetch videos from the specified channel within the date range
videos = get_videos(youtube, channel_id, START_DATE, END_DATE)

# Initialize a list to store all comments
all_comments = []

# Fetch comments for each video
for video in videos:
    video_id = video['id']['videoId']
    try:
        comments = get_comments(youtube, video_id)
        all_comments.extend(comments)
    except HttpError as e:
        print(f"An error occurred: {e}")

# Filter and count keywords in comments
keyword_counts = filter_and_count_comments(all_comments, KEYWORDS, START_DATE, END_DATE)

NameError: name 'build' is not defined

In [ ]:
comments[0]

In [ ]:
# Preparing dataset for visualization
df_keywords = pd.DataFrame(keyword_counts.items()).set_index(0)
df_keywords.index = pd.to_datetime(df_keywords.index)
df_keywords.sort_index(inplace=True)
df_keywords.columns = ['count']
df_keywords['count'] = df_keywords['count'].astype(int)
df_keywords = df_keywords.resample('D').sum()

rolling_sum = df_keywords.loc[:"2020-05-30"].rolling('7D').sum()

# Downloading S&P500 as a benchmark
gspc = yf.download('^GSPC', start='2020-01-01', end='2020-05-30')['Adj Close']

# Plotting
fig, ax1 = plt.subplots(figsize=(16, 7))

# Plotting the keywords data
#ax1.plot(df_keywords.loc[:"2020-05-30"].rolling('7D').sum(), color='green', alpha=0.6, label='Rolling weekly sum of keywords')
ax1.bar(rolling_sum.index, rolling_sum.values.flatten(), color='green', alpha=0.6, label='Rolling weekly sum of keywords')
#ax1.set_ylim(-1, 40)
ax1.set_ylabel('Sum of Keywords')

# Creating a twin axis to plot S&P500 data
ax2 = ax1.twinx()
ax2.plot(gspc, color='blue', alpha=0.5, label='S&P500 Price')
ax2.set_ylim(1500, 3400)
ax2.set_ylabel('S&P500 Price')

# Adding legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Rolling weekly sum of keywords and S&P500 Price')
plt.show()

In [8]:
# Initialize the Reddit instance

reddit = praw.Reddit(
    client_id='CLIENT_ID',
    client_secret="CLIENT_SECRET",
    user_agent='USER_AGENT',
    check_for_async=False
)

# Example of fetching the top posts from a subreddit
submissions = reddit.subreddit('finance').top(limit=5)
for submission in submissions:
    print(f"Title: {submission.title}")
    print(f"Score: {submission.score}")
    print(f"URL: {submission.url}")
    print("="*40)

ResponseException: received 401 HTTP response

In [ ]:
# Define subreddits
subreddits = ['cryptocurrency']

# Define bull and bear sentiment keywords
bull_keywords = ['bull', 'bullish', 'moon', 'lambo', 'hodl', 'buy', 'long', 'uptrend', 'breakout', 'rally']
bear_keywords = ['bear', 'bearish', 'crash', 'dip', 'sell', 'short', 'downtrend', 'correction', 'dump', 'fud']

# Function to get comments from a subreddit
def get_comments(subreddit_name, start_date):
    subreddit = reddit.subreddit(subreddit_name)
    comments = []
    for submission in subreddit.new(limit=None):
        if submission.created_utc < start_date:
            break
        submission.comments.replace_more(limit=0)
        for comment in submission.comments.list():
            comments.append({
                'text': comment.body,
                'created_utc': datetime.fromtimestamp(comment.created_utc),
                'subreddit': subreddit_name
            })
    return comments

# Get comments from the last month
end_date = datetime.now()
start_date = end_date - timedelta(days=30)
all_comments = []

for subreddit in subreddits:
    all_comments.extend(get_comments(subreddit, start_date.timestamp()))

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize

# Download the VADER lexicon
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')

# Initialize the sentiment analyzer
sia = SentimentIntensityAnalyzer()

In [ ]:
# Preparing the dataset for visualization

text_df = pd.DataFrame(all_comments) # Create a DataFrame from the comments

text_df['tokenized'] = text_df['text'].apply(word_tokenize) # Tokenize the comments
text_df['tokenized'] = text_df['tokenized'].apply(lambda x: [word.lower() for word in x]) # Convert words to lowercase
3669
text_df['bull_count'] = text_df['tokenized'].apply(lambda x: sum(1 for word in x if word in bull_keywords)) # Count bull keywords
text_df['bear_count'] = text_df['tokenized'].apply(lambda x: sum(1 for word in x if word in bear_keywords)) # Count bear keywords

text_df['sia'] = text_df['text'].apply(lambda x: sia.polarity_scores(x)['compound']) # Calculate sentiment using NLTK

text_df.set_index('created_utc', inplace=True) # Set the index to the created_utc column
text_df.sort_index(inplace=True) # Sort the DataFrame by the index

In [ ]:
# Downloading BTCUSD price as a cryptomarket proxy.
btcusd = yf.download('BTC-USD', start=start_date, end=end_date)['Adj Close'] # Download Bitcoin price data

In [ ]:
# Superimposing the price on the smoothened sentiment score

fig, ax1 = plt.subplots(figsize=(14, 5)) # Create a figure and axis
ax2 = ax1.twinx() # Create a second axis that shares the same x-axis

smoothened_sia = text_df[['sia']].resample('D').mean().rolling('7D').mean() # Calculate the rolling mean of SIA
ax1.plot(smoothened_sia, color='green', label='Sentiment Analysis Score') # Plot the SIA mean
ax2.plot(btcusd, color='blue', label='BTC-USD') # Plot the Bitcoin price

ax1.set_ylabel('SIA Sentiment') # Set the label for the first axis
ax2.set_ylabel('BTC-USD Price') # Set the label for the second axis
ax1.set_xlabel('Date') # Set the label for the x-axis

# Get the handles and labels from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

# Combine the handles and labels, and create a legend
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.title('SIA Sentiment vs. BTC-USD Price') # Set the title of the plot
plt.show() # Show the plot

In [ ]:
keywords_bull_bear = text_df[['bull_count', 'bear_count']].resample('D').sum()
bull_minus_bear = keywords_bull_bear['bull_count'] - keywords_bull_bear['bear_count']
bull_minus_bear = bull_minus_bear.rolling('7D').mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5)) # Create a figure and axis
ax2 = ax1.twinx() # Create a second axis that shares the same x-axis


ax1.plot(bull_minus_bear, color='green', label='Bull - Bear') # Plot the difference
ax2.plot(btcusd, color='blue', label='BTC-USD') # Plot the Bitcoin price

ax1.set_ylabel('Bull - Bear counts') # Set the label for the first axis
ax2.set_ylabel('BTC-USD Price') # Set the label for the second axis
ax1.set_xlabel('Date') # Set the label for the x-axis

# Get the handles and labels from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

# Combine the handles and labels, and create a legend
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.title('Bull - Bear counts vs. BTC-USD Price') # Set the title of the plot
plt.show() # Show the plot